# Radio cross-match

Cross-match the OVRO-LWA metacatalog against external radio reference catalogs. Matching is most reliable at similar frequency and angular resolution:

| Catalog | Frequency | Beam | Role |
| --- | --- | --- | --- |
| VLSSR | ~73 MHz | 80″ | Blue-band merge QA (similar freq/resolution) |
| [NVSS](https://heasarc.gsfc.nasa.gov/w3browse/all/nvss.html) | 1.4 GHz | 45″ | Known-source association (Condon et al. 1998) |
| [VLASS QL epoch 1](https://cirada.ca/catalogues) | ~3 GHz | ~2.5″ | Known-source association (Gordon et al. 2021, [ApJS 255, 30](https://scixplorer.org/abs/2021ApJS..255...30G/abstract)) |

Reference catalog files live under `/fast/claw/catalogs/` (see `REFERENCE_CATALOGS_DIR` in `lwa_catalog.constants`).

**Matching:** beam-radius association via `associate_catalogs` on metacatalog `RA`/`DEC` and `BMAJ_match`.

**Run cells in order.**

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lwa_catalog import CatalogLayout, read_metacatalog, resolve_metacatalog_path
from lwa_catalog.analyze import (
    NvssMatchConfig,
    VlassMatchConfig,
    VlssrMatchConfig,
    filter_or_hesl,
    load_nvss_catalog,
    load_vlass_catalog,
    load_vlssr_catalog,
    match_catalog_to_nvss,
    match_catalog_to_vlass,
    match_catalog_to_vlssr,
    predict_flux_at_frequency_hz,
    resolve_highest_frequency_peak_flux,
    select_metacatalog,
    summarize_nvss_match,
    summarize_vlass_match,
    summarize_vlssr_match,
)
from lwa_catalog.constants import (
    BAND_FREQ_HZ,
    NVSS_DEFAULT_PATH,
    NVSS_FREQ_HZ,
    REFERENCE_CATALOGS_DIR,
    VLASS_DEFAULT_PATH,
    VLASS_FREQ_HZ,
    VLSSR_DEFAULT_PATH,
)

# --- operator config ---
CATALOG_DIR = Path("/fast/claw/metacatalog_coaddR-0.75")
VLSSR_PATH = VLSSR_DEFAULT_PATH
NVSS_PATH = NVSS_DEFAULT_PATH
VLASS_PATH = VLASS_DEFAULT_PATH
APPLY_OR_HESL_FILTER = True

# Metacatalog subset for NVSS / VLASS cross-match:
#   "full" | "blue" | "quality_all_clear" | "query"
METACATALOG_SELECTION = "blue"
METACATALOG_QUERY = "Peak_flux > 1.0"  # used only when METACATALOG_SELECTION == "query"

# NVSS / VLASS match target: "metacatalog" or "metacatalog_blue"
RADIO_MATCH_TARGET = "metacatalog"

layout = CatalogLayout(CATALOG_DIR)
vlssr_config = VlssrMatchConfig(catalog_path=VLSSR_PATH, target="metacatalog_blue")
nvss_config = NvssMatchConfig(catalog_path=NVSS_PATH, target=RADIO_MATCH_TARGET)
vlass_config = VlassMatchConfig(catalog_path=VLASS_PATH, target=RADIO_MATCH_TARGET)

print("CATALOG_DIR =", layout.root.resolve())
print("metacatalog path =", resolve_metacatalog_path(layout))
print("REFERENCE_CATALOGS_DIR =", REFERENCE_CATALOGS_DIR.resolve())
print("VLSSR_PATH =", Path(VLSSR_PATH).resolve())
print("NVSS_PATH =", Path(NVSS_PATH).resolve())
print("VLASS_PATH =", Path(VLASS_PATH).resolve())
print("APPLY_OR_HESL_FILTER =", APPLY_OR_HESL_FILTER)
print("METACATALOG_SELECTION =", METACATALOG_SELECTION)

## Load and select metacatalog

In [ ]:
metacatalog = read_metacatalog(
    layout,
    prefer_quality=True,
    prefer_spectral=True,
    quality_mask=None,
)
if APPLY_OR_HESL_FILTER:
    metacatalog = filter_or_hesl(metacatalog)
selected_meta = select_metacatalog(
    metacatalog,
    selection=METACATALOG_SELECTION,
    layout=layout,
    query=METACATALOG_QUERY,
)

print(f"metacatalog rows: {len(metacatalog)}")
print(f"selected rows ({METACATALOG_SELECTION}): {len(selected_meta)}")
print(f"spectral columns present: {any(c.startswith('spec_peak_') for c in selected_meta.columns)}")

In [ ]:
def build_unique_flux_table(
    selected_meta: pd.DataFrame,
    meta_flags: pd.DataFrame,
    ref_footprint: pd.DataFrame,
    *,
    count_col: str,
    positions_col: str,
    ref_peak_col: str,
    ref_freq_hz: float,
    lwa_flux_col: str,
) -> pd.DataFrame:
    unique_flags = meta_flags.loc[meta_flags[count_col] == 1].copy()
    meta_by_id = selected_meta.set_index("meta_id", drop=False)
    records: list[dict] = []
    for _, flag in unique_flags.iterrows():
        positions = flag.get(positions_col, [])
        if len(positions) != 1:
            continue
        meta_id = flag["meta_id"]
        if meta_id not in meta_by_id.index:
            continue
        meta_row = meta_by_id.loc[meta_id]
        if isinstance(meta_row, pd.DataFrame):
            meta_row = meta_row.iloc[0]
        ref_row = ref_footprint.iloc[int(positions[0])]
        ref_peak = float(ref_row[ref_peak_col])
        lwa_flux = predict_flux_at_frequency_hz(meta_row, ref_freq_hz, flux_kind="peak")
        records.append(
            {
                "meta_id": meta_id,
                "RA": flag["RA"],
                "DEC": flag["DEC"],
                "ref_peak_jy": ref_peak,
                lwa_flux_col: lwa_flux,
                "log_ratio": np.log10(lwa_flux / ref_peak)
                if np.isfinite(lwa_flux) and ref_peak > 0
                else np.nan,
            }
        )
    return pd.DataFrame.from_records(records)


def plot_unique_flux_check(
    flux_table: pd.DataFrame, *, title: str, lwa_flux_col: str, ref_label: str
) -> None:
    finite = flux_table.dropna(subset=["log_ratio"])
    if finite.empty:
        print("No rows with spectral extrapolation for plot.")
        return
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(
        np.log10(finite["ref_peak_jy"]),
        np.log10(finite[lwa_flux_col]),
        s=8,
        alpha=0.35,
    )
    lim = [
        min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1]),
    ]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set_xlabel(f"log10 {ref_label} peak (Jy)")
    ax.set_ylabel(f"log10 LWA extrapolated peak ({lwa_flux_col})")
    ax.set_title(title)
    plt.show()

## VLSSR cross-match (~73 MHz)

Post-hoc QA against the VLSSR reference catalog at similar frequency and resolution to the Blue band.

**Primary metric:** Blue-associated completeness — fraction of rows with `"Blue" in bands_present` that match ≥1 VLSSR source within beam.

**Failure-mode metric:** VLSSR over-splitting — one VLSSR source matched by multiple metacatalog rows.

In [ ]:
vlssr = load_vlssr_catalog(VLSSR_PATH)
print(f"VLSSR rows: {len(vlssr):,}")
vlssr_result = match_catalog_to_vlssr(metacatalog, vlssr=vlssr, config=vlssr_config)
print(summarize_vlssr_match(vlssr_result))

In [ ]:
vlssr_meta_flags = vlssr_result.meta_flags
vlssr_ref_flags = vlssr_result.vlssr_flags

print("VLSSR hits per meta (value_counts):")
display(vlssr_meta_flags["n_vlssr"].value_counts().sort_index().head(20))

vlssr_incomplete = vlssr_meta_flags.loc[~vlssr_meta_flags["matched"]]
print(f"\nIncomplete Blue-associated rows (no VLSSR match, first 20 of {len(vlssr_incomplete)}):")
display(vlssr_incomplete.head(20))

vlssr_oversplit = vlssr_ref_flags.loc[vlssr_ref_flags["oversplit"]]
print(f"\nVLSSR over-split rows (first 20 of {len(vlssr_oversplit)}):")
display(vlssr_oversplit.head(20))

In [ ]:
blue_freq_hz = BAND_FREQ_HZ["Blue"]
meta_by_id = metacatalog.set_index("meta_id", drop=False)
vlssr_unique = vlssr_meta_flags.loc[vlssr_meta_flags["n_vlssr"] == 1].copy()

vlssr_flux_records: list[dict] = []
for _, flag in vlssr_unique.iterrows():
    positions = flag.get("vlssr_positions", [])
    if len(positions) != 1:
        continue
    meta_id = flag["meta_id"]
    if meta_id not in meta_by_id.index:
        continue
    meta_row = meta_by_id.loc[meta_id]
    if isinstance(meta_row, pd.DataFrame):
        meta_row = meta_row.iloc[0]
    ref_row = vlssr_result.vlssr_footprint.iloc[int(positions[0])]
    ref_peak = float(ref_row["Peak_flux"])
    lwa_flux, _, _ = resolve_highest_frequency_peak_flux(meta_row)
    vlssr_flux_records.append(
        {
            "meta_id": meta_id,
            "vlssr_peak_jy": ref_peak,
            "lwa_peak_blue_jy": lwa_flux,
            "log_ratio": np.log10(lwa_flux / ref_peak)
            if np.isfinite(lwa_flux) and ref_peak > 0
            else np.nan,
        }
    )

vlssr_flux_table = pd.DataFrame.from_records(vlssr_flux_records)
print(f"unique VLSSR matches: {len(vlssr_unique)}")
print(f"with LWA Blue peak flux: {vlssr_flux_table['lwa_peak_blue_jy'].notna().sum()}")
display(vlssr_flux_table.head(20))

finite = vlssr_flux_table.dropna(subset=["log_ratio"])
if not finite.empty:
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(
        np.log10(finite["vlssr_peak_jy"]),
        np.log10(finite["lwa_peak_blue_jy"]),
        s=8,
        alpha=0.35,
    )
    lim = [
        min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1]),
    ]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set_xlabel("log10 VLSSR peak (Jy)")
    ax.set_ylabel("log10 LWA Blue peak (Jy)")
    ax.set_title(f"Unique VLSSR matches @ {blue_freq_hz / 1e6:.0f} MHz")
    plt.show()

## NVSS cross-match (1.4 GHz)

Loads ~1.8M rows from the NRAO FITS table (30–60 s on first read).

In [ ]:
nvss = load_nvss_catalog(NVSS_PATH)
print(f"NVSS rows: {len(nvss):,}")
print(f"Dec range: {nvss['DEC'].min():.2f} .. {nvss['DEC'].max():.2f}")
nvss_result = match_catalog_to_nvss(selected_meta, nvss=nvss, config=nvss_config)
print(summarize_nvss_match(nvss_result))

In [ ]:
nvss_meta_flags = nvss_result.meta_flags
nvss_flags = nvss_result.nvss_flags

print("NVSS hits per meta (value_counts):")
display(nvss_meta_flags["n_nvss"].value_counts().sort_index().head(20))

nvss_unmatched = nvss_meta_flags.loc[~nvss_meta_flags["matched"]]
print(f"\nUnmatched meta rows (first 20 of {len(nvss_unmatched)}):")
display(nvss_unmatched.head(20))

In [ ]:
nvss_flux_table = build_unique_flux_table(
    selected_meta,
    nvss_meta_flags,
    nvss_result.nvss_footprint,
    count_col="n_nvss",
    positions_col="nvss_positions",
    ref_peak_col="Peak_intensity",
    ref_freq_hz=NVSS_FREQ_HZ,
    lwa_flux_col="lwa_peak_1p4ghz_jy",
)
print(f"unique NVSS matches: {(nvss_meta_flags['n_nvss'] == 1).sum()}")
print(f"with spectral extrapolation: {nvss_flux_table['lwa_peak_1p4ghz_jy'].notna().sum()}")
display(nvss_flux_table.head(20))
plot_unique_flux_check(
    nvss_flux_table,
    title="Unique NVSS matches",
    lwa_flux_col="lwa_peak_1p4ghz_jy",
    ref_label="NVSS",
)

## VLASS cross-match (~3 GHz)

Loads the CIRADA QL epoch 1 component table with recommended quality flags (`Duplicate_flag <= 1`, `Quality_flag in (0, 4)`, `S_Code != 'E'`). First read can take several minutes.

In [ ]:
vlass = load_vlass_catalog(VLASS_PATH, config=vlass_config)
print(f"VLASS rows (after quality filter): {len(vlass):,}")
print(f"Dec range: {vlass['DEC'].min():.2f} .. {vlass['DEC'].max():.2f}")
vlass_result = match_catalog_to_vlass(selected_meta, vlass=vlass, config=vlass_config)
print(summarize_vlass_match(vlass_result))

In [ ]:
vlass_meta_flags = vlass_result.meta_flags
vlass_ref_flags = vlass_result.vlass_flags

print("VLASS hits per meta (value_counts):")
display(vlass_meta_flags["n_vlass"].value_counts().sort_index().head(20))

vlass_unmatched = vlass_meta_flags.loc[~vlass_meta_flags["matched"]]
print(f"\nUnmatched meta rows (first 20 of {len(vlass_unmatched)}):")
display(vlass_unmatched.head(20))

vlass_oversplit = vlass_ref_flags.loc[vlass_ref_flags["oversplit"]]
print(f"\nVLASS over-split rows (first 20 of {len(vlass_oversplit)}):")
display(vlass_oversplit.head(20))

In [ ]:
vlass_flux_table = build_unique_flux_table(
    selected_meta,
    vlass_meta_flags,
    vlass_result.vlass_footprint,
    count_col="n_vlass",
    positions_col="vlass_positions",
    ref_peak_col="Peak_flux",
    ref_freq_hz=VLASS_FREQ_HZ,
    lwa_flux_col="lwa_peak_3ghz_jy",
)
print(f"unique VLASS matches: {(vlass_meta_flags['n_vlass'] == 1).sum()}")
print(f"with spectral extrapolation: {vlass_flux_table['lwa_peak_3ghz_jy'].notna().sum()}")
display(vlass_flux_table.head(20))
plot_unique_flux_check(
    vlass_flux_table,
    title="Unique VLASS matches",
    lwa_flux_col="lwa_peak_3ghz_jy",
    ref_label="VLASS",
)